In [1]:
import os
import sys

import langchain
import langchain_core
import langchain_openai

from dotenv import load_dotenv

load_dotenv()

# 가상환경 확인
print(f"Python 실행 경로: {sys.executable}")
if ".venv" in sys.executable:
    
    print("[OK] .venv 가상환경 사용 중")
else:
    print("[WARN] .venv 가상환경이 아닌 인터프리터가 선택되어 있습니다")

print("[OK] OpenAI API 키 설정됨" if os.getenv("OPENAI_API_KEY") else "[ERROR] OpenAI API 키 미설정")
print("[OK] Anthropic API 키 설정됨" if os.getenv("ANTHROPIC_API_KEY") else "[ERROR] Anthropic API 키 미설정")
print("[OK] Google API 키 설정됨" if os.getenv("GOOGLE_API_KEY") else "[ERROR] Google API 키 미설정")
print("[OK] LangSmith 키 설정됨" if os.getenv("LANGSMITH_API_KEY") else "[ERROR] LangSmith 키 미설정")

print(f"LangChain: {langchain.__version__}")
print(f"LangChain Core: {langchain_core.__version__}")
print(f"LangChain OpenAI: {langchain_openai.__version__}")

Python 실행 경로: d:\yugyeong\hanwha_0902\days\ex_0918\.venv\Scripts\python.exe
[OK] .venv 가상환경 사용 중
[OK] OpenAI API 키 설정됨
[ERROR] Anthropic API 키 미설정
[ERROR] Google API 키 미설정
[OK] LangSmith 키 설정됨
LangChain: 1.4.1
LangChain Core: 1.6.3
LangChain OpenAI: 1.6.2


In [4]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_openai import ChatOpenAI

In [5]:
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

In [6]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 Question-Answering 챗봇입니다. 주어진 질문에 대한 답변을 제공해주세요.",),

        MessagesPlaceholder(variable_name="chat_history"),

        ("human", "#Question: {question}"),
    ]
)

In [8]:
chain = prompt | llm | StrOutputParser()

store = {}

In [9]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    print (f"[대화 세션 ID]: {session_id}")
    
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
        
    return store[session_id]

In [12]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

In [13]:
response = chain_with_history.invoke(
    {
        "question": "나의 이름은 테디입니다."
    },
    config={
        "configurable": {
            "session_id": "abc123"
        }
    },
)

print(response)

[대화 세션 ID]: abc123
안녕하세요, 테디님. 만나서 반갑습니다. 이 대화에서는 테디님이라고 부를게요. 무엇을 도와드릴까요?


In [14]:
response = chain_with_history.invoke(
    {
        "question": "내 이름이 뭐라고?"
    },
    config={
        "configurable": {
            "session_id": "abc123"
        }
    },
)

print(response)

[대화 세션 ID]: abc123
테디님이에요.
